In [1]:
import scanpy as sc
import numpy as np
import pandas as pd

In [2]:
file_path = "../data/training_cells.h5ad"

In [3]:
adata = sc.read_h5ad(file_path)

In [4]:
adata

AnnData object with n_obs × n_vars = 17882 × 19226
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sgrna_id', 'sgrna_symbol', 'channel'
    var: 'features'

## Preprocessing by the instruction on Kaggle:
> We start from UMI counts of 19,226 genes. The gene list is defined by Gencode v.46.
Raw counts are first log-normalized:
>
> * Divide the UMI counts in each cell by the total UMI count in that cell
> * Multiply by 10,000
> * Log-transform with log1p (log(x + 1)); logarithm base = 2.
> 
> Normalization is performed on all 19,226 genes to facilitate compatibility with other pre-processed datasets. The normalized data are then subset to the 5,127 genes relevant for the challenge. Finally, expression values for each gene are averaged per perturbation. We use a simple arithmetic mean, disregarding batch information.

In [5]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata, base=2)

### Filtering the genes to predict

In [6]:
submission_path = "../data/sample_submission.csv"

In [7]:
submission = pd.read_csv(submission_path)

In [8]:
target_genes = list(submission.columns)[1:]
adata_target = adata[:, adata.var_names.isin(target_genes)].copy()
adata_target

AnnData object with n_obs × n_vars = 17882 × 5127
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sgrna_id', 'sgrna_symbol', 'channel'
    var: 'features'
    uns: 'log1p'

In [9]:
adata_target.to_df().head(5)

,A1BG,A1CF,AADAC,AAK1,AARS1,AASS,ABCA1,ABCA12,ABCA5,ABCB5,...,ZP3,ZPBP,ZRANB3,ZSCAN18,ZSCAN31,ZSWIM5,ZSWIM6,ZSWIM7,ZWINT,ZYX
AAACCAAAGACGCGAA_ch_1,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,...,1.150484,0.0,0.687009,0.0,0.000000,0.0,0.687009,1.150484,0.687009,0.687009
AAACCAAAGCAAATGA_ch_1,1.415037,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,2.369234,0.874469,0.000000,0.000000
AAACCAAAGCAGTCTA_ch_1,0.627569,0.0,0.0,0.627569,0.447110,0.0,0.000000,0.0,0.0,0.0,...,0.240806,0.0,1.397736,0.0,0.000000,0.0,0.787944,0.627569,0.932262,0.627569
AAACCAAAGGGCATAG_ch_1,0.000000,0.0,0.0,1.340110,0.595092,0.0,1.605151,0.0,0.0,0.0,...,1.015168,0.0,1.340110,0.0,0.000000,0.0,1.340110,0.000000,0.595092,1.015168
AAACCATTCCAATCGA_ch_1,0.616832,0.0,0.0,0.616832,1.648013,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.0,0.616832,0.0,2.070741,0.616832,0.000000,0.616832


## Grouping by perturbation

In [10]:
df = adata_target.to_df()
df['sgrna_symbol'] = adata_target.obs['sgrna_symbol']

### Get the average values
Just like the given `train_data_means.csv` file, make a CSV file with average expression values
> Average expression values of unperturbed cells (non-targeting sgRNA) and cells with 80 selected perturbations. These cells are from the same experiment as the cells in the validation and test datasets. The last row, containing non-targeting in the pert_symbol column, is the baseline expression used for reference. All submissions should contain delta expression relative to this row.

In [11]:
grouped_mean = df.groupby('sgrna_symbol').mean()
print(grouped_mean)

                   A1BG      A1CF     AADAC      AAK1     AARS1      AASS  \
sgrna_symbol                                                                
ACLY           0.311642  0.022102  0.100949  0.378671  0.715857  0.000000   
ALDOA          0.627445  0.043961  0.094335  0.442062  0.658125  0.000000   
APAF1          0.467114  0.030714  0.090910  0.470155  0.732770  0.002820   
ARID2          0.517185  0.018776  0.102561  0.445692  0.687079  0.010236   
BAG1           0.409021  0.034862  0.129969  0.462849  0.819877  0.002999   
...                 ...       ...       ...       ...       ...       ...   
TGFBR2         0.537884  0.015554  0.106994  0.455972  0.723697  0.005621   
USP22          0.443085  0.044874  0.123302  0.288553  0.675256  0.000000   
VEGFA          0.498489  0.009164  0.096523  0.420647  0.753090  0.003990   
WAC            0.481894  0.023404  0.085533  0.476920  0.801459  0.000000   
non-targeting  0.474062  0.025797  0.093244  0.443709  0.737890  0.002591   

### Sanity Check: Validate Custom Means
Validate whether our calculated mean values match the baseline provided by the competition.

* Compare our DIY calculated means directly against the competition's `training_data_means.csv`.

In [12]:
original_df = pd.read_csv("../data/training_data_means.csv", index_col = 0)

In [13]:
grouped_mean.head(5)

,A1BG,A1CF,AADAC,AAK1,AARS1,AASS,ABCA1,ABCA12,ABCA5,ABCB5,...,ZP3,ZPBP,ZRANB3,ZSCAN18,ZSCAN31,ZSWIM5,ZSWIM6,ZSWIM7,ZWINT,ZYX
sgrna_symbol,,,,,,,,,,,,,,,,,,,,,
ACLY,0.311642,0.022102,0.100949,0.378671,0.715857,0.000000,0.118900,0.030792,0.087006,0.003792,...,0.290693,0.002349,0.638608,0.031022,0.036856,0.097390,0.914845,0.647727,0.720461,0.603572
ALDOA,0.627445,0.043961,0.094335,0.442062,0.658125,0.000000,0.200191,0.000000,0.061549,0.000000,...,0.390909,0.000000,0.371624,0.045867,0.087408,0.124600,0.821529,0.801302,0.545344,0.553567
APAF1,0.467114,0.030714,0.090910,0.470155,0.732770,0.002820,0.118370,0.025851,0.137382,0.001402,...,0.359183,0.000000,0.535496,0.009991,0.053445,0.069219,0.855117,0.709241,0.674747,0.667791
ARID2,0.517185,0.018776,0.102561,0.445692,0.687079,0.010236,0.124519,0.016378,0.123456,0.002764,...,0.353383,0.000000,0.557026,0.017552,0.062452,0.094811,0.731613,0.614643,0.629297,0.619404
BAG1,0.409021,0.034862,0.129969,0.462849,0.819877,0.002999,0.125579,0.002677,0.131047,0.008825,...,0.286078,0.000000,0.611761,0.008581,0.052864,0.126885,0.899319,0.714828,0.646969,0.636020


In [14]:
original_df.head(5)

,A1BG,A1CF,AADAC,AAK1,AARS1,AASS,ABCA1,ABCA12,ABCA5,ABCB5,...,ZP3,ZPBP,ZRANB3,ZSCAN18,ZSCAN31,ZSWIM5,ZSWIM6,ZSWIM7,ZWINT,ZYX
pert_symbol,,,,,,,,,,,,,,,,,,,,,
ACLY,0.311642,0.022102,0.100949,0.378671,0.715857,0.000000,0.118900,0.030792,0.087006,0.003792,...,0.290693,0.002349,0.638608,0.031022,0.036856,0.097390,0.914845,0.647727,0.720461,0.603572
ALDOA,0.627445,0.043961,0.094335,0.442062,0.658125,0.000000,0.200191,0.000000,0.061549,0.000000,...,0.390909,0.000000,0.371624,0.045867,0.087408,0.124600,0.821529,0.801302,0.545344,0.553567
APAF1,0.467114,0.030714,0.090910,0.470155,0.732770,0.002820,0.118370,0.025851,0.137382,0.001402,...,0.359183,0.000000,0.535496,0.009991,0.053445,0.069219,0.855117,0.709241,0.674747,0.667791
ARID2,0.517185,0.018776,0.102561,0.445692,0.687079,0.010236,0.124519,0.016378,0.123456,0.002764,...,0.353383,0.000000,0.557026,0.017552,0.062452,0.094811,0.731613,0.614643,0.629297,0.619404
BAG1,0.409021,0.034862,0.129969,0.462849,0.819877,0.002999,0.125579,0.002677,0.131047,0.008825,...,0.286078,0.000000,0.611761,0.008581,0.052864,0.126885,0.899319,0.714828,0.646969,0.636020


In [15]:
grouped_mean.index.name = original_df.index.name

In [16]:
original_df.compare(grouped_mean)

A1BG                A1CF               AADAC            \
                   self     other      self     other      self     other   
pert_symbol                                                                 
ACLY           0.311642  0.311642  0.022102  0.022102  0.100949  0.100949   
ALDOA          0.627445  0.627445       NaN       NaN       NaN       NaN   
APAF1          0.467114  0.467114  0.030714  0.030714  0.090910  0.090910   
ARID2          0.517185  0.517185  0.018776  0.018776  0.102561  0.102561   
BAG1           0.409021  0.409021  0.034862  0.034862  0.129969  0.129969   
...                 ...       ...       ...       ...       ...       ...   
TGFBR2         0.537884  0.537884  0.015554  0.015554  0.106994  0.106994   
USP22          0.443085  0.443085  0.044874  0.044874  0.123302  0.123302   
VEGFA          0.498489  0.498489  0.009164  0.009164  0.096523  0.096523   
WAC            0.481894  0.481894  0.023404  0.023404  0.085533  0.085533   
non-targeting  0.474062  0.474062  0.025797  0.025797  0.093244  0.093244   

                   AAK1               AARS1            ...    ZSWIM5  \
                   self     other      self     other  ...      self   
pert_symbol                                            ...             
ACLY           0.378671  0.378671  0.715857  0.715857  ...       NaN   
ALDOA          0.442062  0.442062  0.658125  0.658125  ...  0.124600   
APAF1          0.470155  0.470155       NaN       NaN  ...  0.069219   
ARID2          0.445692  0.445692  0.687079  0.687079  ...  0.094811   
BAG1           0.462849  0.462849  0.819877  0.819877  ...  0.126885   
...                 ...       ...       ...       ...  ...       ...   
TGFBR2         0.455972  0.455972  0.723697  0.723697  ...  0.077293   
USP22          0.288553  0.288553  0.675256  0.675256  ...  0.076007   
VEGFA          0.420647  0.420647  0.753090  0.753090  ...  0.095018   
WAC            0.476920  0.476920  0.801459  0.801459  ...  0.125091   
non-targeting  0.443709  0.443709  0.737890  0.737890  ...  0.078303   

                           ZSWIM6              ZSWIM7               ZWINT  \
                  other      self     other      self     other      self   
pert_symbol                                                                 
ACLY                NaN  0.914845  0.914845       NaN       NaN  0.720461   
ALDOA          0.124600       NaN       NaN  0.801302  0.801302  0.545344   
APAF1          0.069219  0.855117  0.855117  0.709241  0.709241  0.674747   
ARID2          0.094811  0.731613  0.731613  0.614643  0.614643  0.629297   
BAG1           0.126885  0.899319  0.899319  0.714828  0.714828  0.646969   
...                 ...       ...       ...       ...       ...       ...   
TGFBR2         0.077293  0.936977  0.936977  0.731447  0.731447  0.690297   
USP22          0.076007  1.033585  1.033585  0.765131  0.765131  0.633493   
VEGFA          0.095018  0.821152  0.821152  0.698710  0.698710  0.699610   
WAC            0.125091  0.846153  0.846153  0.693998  0.693998  0.703783   
non-targeting  0.078303  0.860658  0.860658  0.719807  0.719807  0.707353   

                              ZYX            
                  other      self     other  
pert_symbol                                  
ACLY           0.720461  0.603572  0.603572  
ALDOA          0.545344  0.553567  0.553567  
APAF1          0.674747  0.667791  0.667791  
ARID2          0.629297  0.619404  0.619404  
BAG1           0.646969  0.636020  0.636020  
...                 ...       ...       ...  
TGFBR2         0.690297  0.639862  0.639862  
USP22          0.633493  0.659664  0.659664  
VEGFA          0.699610       NaN       NaN  
WAC            0.703783  0.726951  0.726951  
non-targeting  0.707353  0.681142  0.681142  

[81 rows x 10250 columns]

`pandas.testing.assert_frame_equal` allows for minor floating-point precision differences and ignores data type mismatches (e.g., `int64` vs `float64`). 

If the cell below runs without throwing an `AssertionError`, it confirms that this preprocessing and grouping logic is perfectly aligned with the competition's standard! 

In [17]:
from pandas.testing import assert_frame_equal
assert_frame_equal(original_df, grouped_mean, check_exact=False, atol=1e-4, check_dtype=False)

In [18]:
grouped_mean.to_csv("../data/train_grouped_mean.csv")